In [6]:
import numpy as np
import time


class BaseLinearRegression:

    def __init__(self):

        self.theta = None
        self.training_time = None

    def _add_bias(self, X):

        return np.c_[np.ones((X.shape[0], 1)), X]

    def predict(self, X):

        X_b = self._add_bias(X)

        return X_b @ self.theta


# =========================================================
# Closed Form Solution
# =========================================================

class LinearRegressionCFS(BaseLinearRegression):

    """
    Closed Form Solution

    theta = (X^T X)^-1 X^T y
    """

    def fit(self, X, y):

        start_time = time.perf_counter()

        X_b = self._add_bias(X)

        self.theta = (
            np.linalg.inv(X_b.T @ X_b)
            @ X_b.T
            @ y
        )

        end_time = time.perf_counter()

        self.training_time = end_time - start_time

        return self


# =========================================================
# SVD Solution
# =========================================================

class LinearRegressionSVD(BaseLinearRegression):

    """
    Linear Regression using SVD
    """

    def fit(self, X, y):

        start_time = time.perf_counter()

        X_b = self._add_bias(X)

        U, S, VT = np.linalg.svd(
            X_b,
            full_matrices=False
        )

        S_inv = np.diag(1 / S)

        X_pseudo_inv = VT.T @ S_inv @ U.T

        self.theta = X_pseudo_inv @ y

        end_time = time.perf_counter()

        self.training_time = end_time - start_time

        return self


# =========================================================
# Gradient Descent
# =========================================================

class LinearRegressionGD(BaseLinearRegression):

    def __init__(
        self,
        learning_rate=0.1,
        n_iterations=1000
    ):

        super().__init__()

        self.learning_rate = learning_rate
        self.n_iterations = n_iterations

    def fit(self, X, y):

        start_time = time.perf_counter()

        X_b = self._add_bias(X)

        m, n = X_b.shape

        self.theta = np.random.randn(n, 1)

        for _ in range(self.n_iterations):

            gradients = (
                (2 / m)
                * X_b.T
                @ (X_b @ self.theta - y)
            )

            self.theta -= (
                self.learning_rate
                * gradients
            )

        end_time = time.perf_counter()

        self.training_time = end_time - start_time

        return self


# =========================================================
# Mini Batch Gradient Descent
# =========================================================

class LinearRegressionMBGD(BaseLinearRegression):

    def __init__(
        self,
        learning_rate=0.1,
        n_epochs=50,
        batch_size=16
    ):

        super().__init__()

        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.batch_size = batch_size

    def fit(self, X, y):

        start_time = time.perf_counter()

        X_b = self._add_bias(X)

        m, n = X_b.shape

        self.theta = np.random.randn(n, 1)

        for _ in range(self.n_epochs):

            indices = np.random.permutation(m)

            X_shuffled = X_b[indices]
            y_shuffled = y[indices]

            for i in range(
                0,
                m,
                self.batch_size
            ):

                xi = X_shuffled[
                    i:i+self.batch_size
                ]

                yi = y_shuffled[
                    i:i+self.batch_size
                ]

                gradients = (
                    (2 / len(xi))
                    * xi.T
                    @ (xi @ self.theta - yi)
                )

                self.theta -= (
                    self.learning_rate
                    * gradients
                )

        end_time = time.perf_counter()

        self.training_time = end_time - start_time

        return self

In [7]:
# =========================================================
# Generate Data
# =========================================================

np.random.seed(42)

X = 2 * np.random.rand(1000, 1)

y = (
    4
    + 3 * X
    + np.random.randn(1000, 1)
)

# =========================================================
# Train Models
# =========================================================

models = {
    "Closed Form": LinearRegressionCFS(),
    "SVD": LinearRegressionSVD(),
    "Gradient Descent": LinearRegressionGD(),
    "Mini Batch GD": LinearRegressionMBGD()
}

for name, model in models.items():

    model.fit(X, y)

    predictions = model.predict(X)

    print(f"\n{name}")

    print("-" * 40)

    print("Theta:")
    print(model.theta.ravel())

    print(
        f"Training Time: "
        f"{model.training_time:.6f} seconds"
    )


Closed Form
----------------------------------------
Theta:
[4.17478026 2.92260742]
Training Time: 0.125639 seconds

SVD
----------------------------------------
Theta:
[4.17478026 2.92260742]
Training Time: 0.029201 seconds

Gradient Descent
----------------------------------------
Theta:
[4.17478026 2.92260742]
Training Time: 0.034060 seconds

Mini Batch GD
----------------------------------------
Theta:
[4.03796932 2.77119068]
Training Time: 0.045605 seconds
